In [3]:
import pandas as pd

df_ground_truth = pd.read_csv("data/ground_truth-new.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [4]:
ground_truth

[{'question': 'Can I still join the course if I found it late?',
  'document': '74eb249bbf'},
 {'question': 'Am I allowed to enroll after the course has already started?',
  'document': '74eb249bbf'},
 {'question': 'If I join now, can I still get a certificate?',
  'document': '74eb249bbf'},
 {'question': 'What do I need to do to qualify for the certificate if I join late?',
  'document': '74eb249bbf'},
 {'question': 'Is it too late to participate, or can I still submit the project?',
  'document': '74eb249bbf'},
 {'question': 'Do I actually need a confirmation email after registering, or can I just start the LLM Zoomcamp right away?',
  'document': '977bf7786c'},
 {'question': 'If I signed up for the course, does that mean I’m already accepted?',
  'document': '977bf7786c'},
 {'question': 'Can I begin watching the lessons and submitting homework even if I never got any email back?',
  'document': '977bf7786c'},
 {'question': 'Is registration for the LLM Zoomcamp used to approve studen

In [5]:
from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

In [6]:
doc_idx = {}

for doc in documents:
    doc_idx[doc["id"]] = doc

In [8]:
doc_idx

{'74eb249bbf': {'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 '977bf7786c': {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 '489dd1c9d9': {'id': '489dd1c9d9',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'What is the video/zoom link to the stream for the “Office Hours

In [9]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

Import RAGWithUsage

Using parameters question=1.0, answer=4.0, and section=0.2

In [10]:
from evaluation_utils import RAGWithUsage

assistant = RAGWithUsage(
    index=index,
    llm_client=openai_client,
)

In [11]:
ground_truth[0]

{'question': 'Can I still join the course if I found it late?',
 'document': '74eb249bbf'}

In [12]:
rec = ground_truth[0]
question = rec["question"]

answer_llm = assistant.rag(question)
answer_llm

'Yes, you can still join the course if you found it late. If you want to receive a certificate, make sure to submit your project while submissions are still being accepted.'

In [13]:
assistant.total_cost()

0.0005445000000000001

In [14]:
doc_id = rec["document"]
original_doc = doc_idx[doc_id]
answer_orig = original_doc["answer"]

answer_orig

'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'

In [15]:
rag_result = {
    "question": question,
    "answer_llm": answer_llm,
    "answer_orig": answer_orig,
    "document": doc_id,
}

rag_result

{'question': 'Can I still join the course if I found it late?',
 'answer_llm': 'Yes, you can still join the course if you found it late. If you want to receive a certificate, make sure to submit your project while submissions are still being accepted.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'document': '74eb249bbf'}

In [16]:
def generate_rag_answer(rec):
    question = rec["question"]
    doc_id = rec["document"]
    original_doc = doc_idx[doc_id]

    answer_llm = assistant.rag(question)
    answer_orig = original_doc["answer"]

    result = {
        "question": question,
        "answer_llm": answer_llm,
        "answer_orig": answer_orig,
        "document": doc_id,
    }

    return result

In [17]:
answer_record = generate_rag_answer(ground_truth[0])
answer_record

{'question': 'Can I still join the course if I found it late?',
 'answer_llm': 'Yes, you can still join the course if you found it late. If you want to receive a certificate, make sure to submit your project while submissions are still being accepted.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'document': '74eb249bbf'}

In [18]:
assistant.reset_usage()

In [19]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [20]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, ground_truth, generate_rag_answer)

  0%|          | 0/575 [00:00<?, ?it/s]

In [21]:
answers = []

for answer_record in results:
    answers.append(answer_record)

In [22]:
assistant.total_cost()

0.6222937500000002

In [23]:
df_answers = pd.DataFrame(answers)
df_answers.to_csv("data/rag-answers-new.csv", index=False)

In [24]:
df_answers.head(5)

,question,answer_llm,answer_orig,document
0,Can I still join the course if I found it late?,"Yes, you can still join the course if you foun...","Yes, but if you want to receive a certificate,...",74eb249bbf
1,Am I allowed to enroll after the course has al...,I don't know.,"Yes, but if you want to receive a certificate,...",74eb249bbf
2,"If I join now, can I still get a certificate?","Yes, but only if you finish with the live coho...","Yes, but if you want to receive a certificate,...",74eb249bbf
3,What do I need to do to qualify for the certif...,"If you join late, you can still qualify for th...","Yes, but if you want to receive a certificate,...",74eb249bbf
4,"Is it too late to participate, or can I still ...","Yes, you can still participate **as long as th...","Yes, but if you want to receive a certificate,...",74eb249bbf


A->Q->A' evaluation

In [25]:
from pydantic import BaseModel, Field
from typing import Literal

class AnswerEvaluation(BaseModel):
    reasoning: str = Field(
        description="Reasoning about the quality of the answer."
    )
    score: Literal["good", "bad"] = Field(
        description="'good' if the answer is correct and complete, 'bad' otherwise."
    )

In [26]:
aqa_judge_instructions = """
You are an expert evaluator. You will be given:
1. A question from a student
2. The original answer from the FAQ (ground truth)
3. An answer generated by an AI assistant

Your task is to decide if the AI answer is semantically equivalent to
the original answer.

Rules:
- The AI answer does NOT need to be word-for-word identical
- It should convey the same key information
- Extra detail is fine as long as the core answer is correct
- Mark 'bad' only if the AI answer is wrong or misses the key point

Be fair and focus on correctness, not style.
""".strip()

In [27]:
aqa_judge_prompt = """
Question:
{question}

Original Answer (ground truth):
{answer_orig}

AI Answer:
{answer_llm}
""".strip()

In [28]:
from dotenv import load_dotenv
from openai import OpenAI
from evaluation_utils import calc_price, calc_total_price, llm_structured_retry, map_progress

load_dotenv()
openai_client = OpenAI()

In [29]:
rec = answers[0]

In [30]:
rec

{'question': 'Can I still join the course if I found it late?',
 'answer_llm': 'Yes, you can still join the course if you found it late. If you want to receive a certificate, make sure to submit your project while submissions are still open.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'document': '74eb249bbf'}

In [31]:
prompt = aqa_judge_prompt.format(
    question=rec["question"],
    answer_orig=rec["answer_orig"],
    answer_llm=rec["answer_llm"]
)

In [32]:
eval_result, usage = llm_structured_retry(
    openai_client,
    aqa_judge_instructions,
    prompt,
    AnswerEvaluation,
)

eval_result

AnswerEvaluation(reasoning='The AI answer preserves the full meaning of the ground truth: late joining is allowed, and certificate eligibility depends on submitting the project before submissions close. It is semantically equivalent.', score='good')

In [33]:
calc_price(usage)

{'input_cost': 0.00022275000000000002,
 'output_cost': 0.0002295,
 'total_cost': 0.00045225}